In [0]:
import csv
import json
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker
from scipy.stats import binomtest

# ── Data loading ──────────────────────────────────────────────────────────────

def read_csv(path):
    with open(path, newline='', encoding='utf-8') as f:
        return list(csv.DictReader(f))

def to_acc(val):
    return 1 if str(val).strip() in ('1', '1.0', 'True', 'true') else 0

# DASE seeds — first 30 rows used for all files (AIME 2026 subset)
dase_seeds = [
    [to_acc(r['DASE_Acc']) for r in read_csv("DASE_Bellman_R30AIME_ai_2_L15_alpha0_2.csv")],
    [to_acc(r['DASE_Acc']) for r in read_csv("DASE_Bellman_R300AIME_ai_2_L15_alpha0_2.csv")[:30]],
    [to_acc(r['DASE_Acc']) for r in read_csv("DASE_Bellman_R300AIME_ai2_2_L15_alpha0_2.csv")[:30]],
]
s1_seeds = [
    [to_acc(r['S1_Acc']) for r in read_csv("DASE_Bellman_R30AIME_ai_2_L15_alpha0_2.csv")],
    [to_acc(r['S1_Acc']) for r in read_csv("DASE_Bellman_R300AIME_ai_2_L15_alpha0_2.csv")[:30]],
    [to_acc(r['S1_Acc']) for r in read_csv("DASE_Bellman_R300AIME_ai2_2_L15_alpha0_2.csv")[:30]],
]
op_std_seeds = [
    [to_acc(r['FM_Acc']) for r in read_csv(f"claude-opus-4-6_standard_AIME2026_{i}.csv")]
    for i in range(1, 4)
]
op_high_seeds = [
    [to_acc(r['FM_Acc']) for r in read_csv(f"Single_FM_Baseline_claude-opus-4-6_adaptive_high_AIME2026_{i}.csv")]
    for i in range(1, 4)
]

# ── Statistics ────────────────────────────────────────────────────────────────

np.random.seed(42)

def bootstrap_ci(seeds, n_boot=50000):
    """Bootstrap 95% CI by resampling problems within each seed, then averaging across seeds."""
    boot_means = [
        np.mean([np.mean(np.random.choice(s, len(s), replace=True)) for s in seeds])
        for _ in range(n_boot)
    ]
    m = np.mean([np.mean(s) for s in seeds])
    return m, np.percentile(boot_means, 2.5), np.percentile(boot_means, 97.5)

def mcnemar_pooled(seeds_a, seeds_b):
    """McNemar exact test pooled across seeds (N = n_seeds × n_problems pairs)."""
    n10 = n01 = 0
    for sa, sb in zip(seeds_a, seeds_b):
        for a, b in zip(sa, sb):
            if a == 1 and b == 0: n10 += 1
            elif a == 0 and b == 1: n01 += 1
    disc = n10 + n01
    if disc == 0:
        return 1.0, n10, n01
    p = binomtest(n10, disc, 0.5, alternative='two-sided').pvalue
    return p, n10, n01

def sig_label(p):
    if   p < 0.001: return "*** p<0.001"
    elif p < 0.01:  return f"** p={p:.3f}"
    elif p < 0.05:  return f"* p={p:.3f}"
    else:           return f"ns p={p:.3f}"

all_seeds = {
    "Opus Standard": op_std_seeds,
    "S1 Consensus":  s1_seeds,
    "Opus High":     op_high_seeds,
    "DASE W=2":      dase_seeds,
}

stats_vs_dase = {
    name: mcnemar_pooled(dase_seeds, seeds)
    for name, seeds in all_seeds.items()
    if name != "DASE W=2"
}

# ── Plot data ─────────────────────────────────────────────────────────────────

keys   = ["Opus Standard", "S1 Consensus", "Opus High", "DASE W=2"]
labels = ["Opus 4.6\nStandard", "S1 Consensus\n(DASE Rd 1)", "Opus 4.6\nAdaptive High", "DASE-Spatial\n(W=2)"]
colors = ["#5B9BD5", "#A8C8E8", "#ED7D31", "#70AD47"]

means     = []
lo_err    = []
hi_err    = []
seed_accs = []

for k in keys:
    m, lo, hi = bootstrap_ci(all_seeds[k])
    means.append(m)
    lo_err.append(m - lo)
    hi_err.append(hi - m)
    seed_accs.append([np.mean(s) for s in all_seeds[k]])

# ── Figure ────────────────────────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(9, 6.5))
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

x     = np.arange(len(labels))
bar_w = 0.52

bars = ax.bar(x, means, bar_w, color=colors, zorder=3,
              linewidth=0.8, edgecolor='white')

ax.errorbar(x, means,
            yerr=[lo_err, hi_err],
            fmt='none', color='#333333',
            capsize=5, capthick=1.4, elinewidth=1.4, zorder=4)

rng = np.random.default_rng(0)
for i, accs in enumerate(seed_accs):
    jitter = rng.uniform(-0.10, 0.10, len(accs))
    ax.scatter(x[i] + jitter, accs, color='white', s=42,
               edgecolors=colors[i], linewidths=1.6, zorder=5)

for bar, m in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width() / 2, m - 0.032,
            f"{m:.1%}", ha='center', va='top',
            fontsize=13, fontweight='bold', color='white', zorder=6)

# ── Significance brackets ─────────────────────────────────────────────────────

def draw_bracket(ax, x1, x2, y, label, color='#555555', lw=1.2):
    h = 0.012
    ax.plot([x1, x1, x2, x2], [y, y + h, y + h, y], lw=lw, color=color, zorder=5)
    ax.text((x1 + x2) / 2, y + h + 0.004, label,
            ha='center', va='bottom', fontsize=9, color=color)

top = max(m + e for m, e in zip(means, hi_err)) + 0.04

p_std,  *_ = stats_vs_dase["Opus Standard"]
p_s1,   *_ = stats_vs_dase["S1 Consensus"]
p_high, *_ = stats_vs_dase["Opus High"]

draw_bracket(ax, 0, 3, top + 0.000, sig_label(p_std),  color='#C00000', lw=1.4)
draw_bracket(ax, 1, 3, top + 0.055, sig_label(p_s1),   color='#555555', lw=1.2)
draw_bracket(ax, 2, 3, top + 0.110, sig_label(p_high),  color='#555555', lw=1.2)

ax.axhline(means[3], color=colors[3], linewidth=0.8, linestyle='--', alpha=0.4, zorder=2)

# ── Axes ──────────────────────────────────────────────────────────────────────

ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=11.5)
ax.set_ylabel("Mean Accuracy (N=30)", fontsize=12)
ax.set_ylim(0.30, top + 0.175)
ax.yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(xmax=1, decimals=0))
ax.set_yticks(np.arange(0.30, 1.01, 0.10))
ax.tick_params(axis='both', labelsize=10.5)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', linestyle='--', linewidth=0.5, alpha=0.5, zorder=0)

ax.set_title(
    "AIME 2026 (N=30) · 3 seeds per method\n"
    "Open-weight DASE ensemble vs. Claude Opus 4.6",
    fontsize=13, fontweight='bold', pad=12
)

fig.text(0.5, -0.02,
         "Error bars: 95% bootstrap CI · Dots: individual seeds\n"
         "McNemar exact test pooled across 3 seeds (N=90 pairs)\n"
         "DASE: 3×GPT-OSS-120B + 2×Qwen3-80B-A3B · ~3 rounds avg",
         ha='center', va='top', fontsize=8.5, color='#555555', style='italic')

plt.tight_layout()
plt.savefig('AIME2026_comparison.pdf', dpi=300, bbox_inches='tight', facecolor='white')


In [0]:
"""
DASE-Spatial (W=2) vs Claude Opus 4.6 Standard — AIME 2010-2026 (N=300)
3 seeds each · bootstrapped 95% CI · McNemar exact test (pooled)

Usage:
    python analyze_aime300.py

Expected files in working directory:
    DASE_Bellman_R300AIME_ai_2_L15_alpha0_2.csv
    DASE_Bellman_R300AIME_ai2_2_L15_alpha0_2.csv
    DASE_Bellman_R300AIME_ai3_2_L15_alpha0_2.csv
    DASE_Bellman_H300AIME_ai_2_L15_alpha0_2.jsonl
    DASE_Bellman_H300AIME_ai2_2_L15_alpha0_2.jsonl
    DASE_Bellman_H300AIME_ai3_2_L15_alpha0_2.jsonl
    Opus_AIME300_1.csv
    Opus_AIME300_2.csv
    Opus_AIME300_3.csv
"""

import csv
import json
import numpy as np
from scipy.stats import binomtest

# ── File paths ────────────────────────────────────────────────────────────────

DASE_CSV = [
    "DASE_Bellman_R300AIME_ai_2_L15_alpha0_2.csv",
    "DASE_Bellman_R300AIME_ai2_2_L15_alpha0_2.csv",
    "DASE_Bellman_R300AIME_ai3_2_L15_alpha0_2.csv",
]
DASE_JSONL = [
    "DASE_Bellman_H300AIME_ai_2_L15_alpha0_2.jsonl",
    "DASE_Bellman_H300AIME_ai2_2_L15_alpha0_2.jsonl",
    "DASE_Bellman_H300AIME_ai3_2_L15_alpha0_2.jsonl",
]
OPUS_CSV = [
    "Opus_AIME300_1.csv",
    "Opus_AIME300_2.csv",
    "Opus_AIME300_3.csv",
]

# ── Loaders ───────────────────────────────────────────────────────────────────

def read_csv(path):
    with open(path, newline='', encoding='utf-8') as f:
        return list(csv.DictReader(f))

def read_jsonl(path):
    records = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return records

def to_acc(val):
    return 1 if str(val).strip() in ('1', '1.0', 'True', 'true') else 0

# ── Load data ─────────────────────────────────────────────────────────────────

dase_seeds  = [[to_acc(r['DASE_Acc'])  for r in read_csv(p)] for p in DASE_CSV]
s1_seeds    = [[to_acc(r['S1_Acc'])    for r in read_csv(p)] for p in DASE_CSV]
opus_seeds  = [[to_acc(r['FM_Acc'])    for r in read_csv(p)] for p in OPUS_CSV]
jsonl_seeds = [read_jsonl(p) for p in DASE_JSONL]

N = len(dase_seeds[0])
assert all(len(s) == N for s in dase_seeds + s1_seeds + opus_seeds), \
    "Seed lengths must all match"
print(f"Loaded {N} problems × {len(dase_seeds)} seeds per method\n")

# ── Per-seed accuracies ───────────────────────────────────────────────────────

def seed_accs(seeds):
    return [np.mean(s) for s in seeds]

# ── Bootstrap 95% CI ─────────────────────────────────────────────────────────

np.random.seed(42)

def bootstrap_ci(seeds, n_boot=50_000):
    """
    Resample problems within each seed, average across seeds.
    Returns (mean, lo, hi).
    """
    boot = [
        np.mean([np.mean(np.random.choice(s, len(s), replace=True))
                 for s in seeds])
        for _ in range(n_boot)
    ]
    m = np.mean([np.mean(s) for s in seeds])
    return m, np.percentile(boot, 2.5), np.percentile(boot, 97.5)

# ── McNemar exact test (pooled across seeds) ─────────────────────────────────

def mcnemar_pooled(seeds_a, seeds_b):
    """
    Pool paired (problem, seed) comparisons.
    Returns (p_value, n_a_wins, n_b_wins, n_pairs).
    """
    n10 = n01 = 0
    for sa, sb in zip(seeds_a, seeds_b):
        for a, b in zip(sa, sb):
            if a == 1 and b == 0:
                n10 += 1
            elif a == 0 and b == 1:
                n01 += 1
    disc = n10 + n01
    n_pairs = len(seeds_a) * N
    if disc == 0:
        return 1.0, n10, n01, n_pairs
    p = binomtest(n10, disc, 0.5, alternative='two-sided').pvalue
    return p, n10, n01, n_pairs

def sig_stars(p):
    if   p < 0.001: return "***"
    elif p < 0.01:  return "**"
    elif p < 0.05:  return "*"
    else:           return "ns"

# ── Compute and print ─────────────────────────────────────────────────────────

methods = {
    "DASE W=2":      dase_seeds,
    "S1 Consensus":  s1_seeds,
    "Opus Standard": opus_seeds,
}

results = {}
print("=" * 65)
print(f"{'Method':<18} {'Mean':>7} {'95% CI':>22} {'Seeds'}")
print("=" * 65)
for name, seeds in methods.items():
    m, lo, hi = bootstrap_ci(seeds)
    results[name] = (m, lo, hi)
    sa = seed_accs(seeds)
    print(f"{name:<18} {m:>6.1%}  [{lo:.1%}, {hi:.1%}]"
          f"  {[f'{x:.1%}' for x in sa]}")

print()
print("=" * 65)
print("McNemar exact test (pooled 3 seeds, N=900 pairs)")
print("=" * 65)

comparisons = [
    ("DASE W=2", "Opus Standard"),
    ("DASE W=2", "S1 Consensus"),
    ("S1 Consensus", "Opus Standard"),
]
for ka, kb in comparisons:
    p, n10, n01, npairs = mcnemar_pooled(methods[ka], methods[kb])
    stars = sig_stars(p)
    print(f"  {ka} vs {kb}")
    print(f"    p={p:.4f} {stars}  |  {ka} wins={n10}, {kb} wins={n01}  "
          f"(of {npairs} pairs)")

# ── Per-problem win/loss breakdown ────────────────────────────────────────────

print()
print("=" * 65)
print("Per-problem agreement (averaged across seeds)")
print("=" * 65)

dase_mean_per_prob  = np.mean(dase_seeds, axis=0)   # shape (300,)
opus_mean_per_prob  = np.mean(opus_seeds, axis=0)
s1_mean_per_prob    = np.mean(s1_seeds,   axis=0)

# Classify each problem by majority outcome across seeds
dase_correct = dase_mean_per_prob >= 0.5
opus_correct = opus_mean_per_prob >= 0.5

both_correct   = np.sum(dase_correct & opus_correct)
dase_only      = np.sum(dase_correct & ~opus_correct)
opus_only      = np.sum(~dase_correct & opus_correct)
both_wrong     = np.sum(~dase_correct & ~opus_correct)

print(f"  Both correct:       {both_correct:3d} / {N}")
print(f"  DASE only correct:  {dase_only:3d} / {N}")
print(f"  Opus only correct:  {opus_only:3d} / {N}")
print(f"  Both wrong:         {both_wrong:3d} / {N}")

# ── Infra noise and commit reason breakdown ───────────────────────────────────

print()
print("=" * 65)
print("DASE commit reason breakdown (pooled across 3 seeds)")
print("=" * 65)

commit_counts = {}
infra_rates   = []
steps_all     = []

for csv_path in DASE_CSV:
    for row in read_csv(csv_path):
        cr = row.get('Commit_Reason', 'unknown')
        commit_counts[cr] = commit_counts.get(cr, 0) + 1
        infra_rates.append(float(row.get('Infra_Noise_Rate', 0)))
        steps_all.append(int(row.get('Steps', 0)))

total_problems = sum(commit_counts.values())
for reason, count in sorted(commit_counts.items(),
                             key=lambda x: -x[1]):
    print(f"  {reason:<25} {count:4d}  ({count/total_problems:.1%})")

print(f"\n  Avg infra noise rate:  {np.mean(infra_rates):.1%}")
print(f"  Avg steps to commit:   {np.mean(steps_all):.2f}")
print(f"  Avg inferences:        {np.mean(steps_all)*5:.1f}")

# ── Accuracy by commit reason ─────────────────────────────────────────────────

print()
print("=" * 65)
print("DASE accuracy by commit reason (pooled across 3 seeds)")
print("=" * 65)

commit_acc = {}
for csv_path in DASE_CSV:
    for row in read_csv(csv_path):
        cr  = row.get('Commit_Reason', 'unknown')
        acc = to_acc(row['DASE_Acc'])
        if cr not in commit_acc:
            commit_acc[cr] = []
        commit_acc[cr].append(acc)

for reason in sorted(commit_acc, key=lambda x: -len(commit_acc[x])):
    accs = commit_acc[reason]
    print(f"  {reason:<25} n={len(accs):4d}  acc={np.mean(accs):.1%}")

# ── High-infra-noise problem analysis ────────────────────────────────────────

print()
print("=" * 65)
print("Problems with avg infra noise > 30% (any seed)")
print("=" * 65)

noisy_problems = set()
for csv_path in DASE_CSV:
    for i, row in enumerate(read_csv(csv_path)):
        if float(row.get('Infra_Noise_Rate', 0)) > 0.30:
            noisy_problems.add(i)

print(f"  {len(noisy_problems)} problems with high noise in at least 1 seed")
if noisy_problems:
    noisy_dase = np.mean([dase_mean_per_prob[i] for i in noisy_problems])
    noisy_opus = np.mean([opus_mean_per_prob[i]  for i in noisy_problems])
    print(f"  DASE acc on these:  {noisy_dase:.1%}")
    print(f"  Opus acc on these:  {noisy_opus:.1%}")

# ── AIME 2026 subset (first 30) ───────────────────────────────────────────────

print()
print("=" * 65)
print("AIME 2026 subset (first 30 problems of 300)")
print("=" * 65)

dase_2026  = [s[:30] for s in dase_seeds]
opus_2026  = [s[:30] for s in opus_seeds]
s1_2026    = [s[:30] for s in s1_seeds]

for name, seeds in [("DASE W=2", dase_2026),
                     ("S1 Consensus", s1_2026),
                     ("Opus Standard", opus_2026)]:
    m, lo, hi = bootstrap_ci(seeds)
    print(f"  {name:<18} {m:.1%} [{lo:.1%}, {hi:.1%}]")

p26, n10_26, n01_26, _ = mcnemar_pooled(dase_2026, opus_2026)
print(f"\n  McNemar DASE vs Opus (2026 subset, N=90 pairs):")
print(f"    p={p26:.4f} {sig_stars(p26)}  DASE wins={n10_26}, Opus wins={n01_26}")

# ── Summary table ─────────────────────────────────────────────────────────────

print()
print("=" * 65)
print("SUMMARY")
print("=" * 65)
dm, dlo, dhi = results["DASE W=2"]
om, olo, ohi = results["Opus Standard"]
p_main, *_  = mcnemar_pooled(dase_seeds, opus_seeds)

print(f"  DASE W=2:      {dm:.1%} [{dlo:.1%}, {dhi:.1%}]")
print(f"  Opus Standard: {om:.1%} [{olo:.1%}, {ohi:.1%}]")
print(f"  Gap:           +{dm-om:.1%} in favour of DASE")
print(f"  McNemar:       p={p_main:.4f} {sig_stars(p_main)}")
print(f"  N problems:    {N}  |  N seeds:  {len(dase_seeds)}")
print(f"  N pairs:       {N * len(dase_seeds)}")

Loaded 300 problems × 3 seeds per method

Method                Mean                 95% CI Seeds
DASE W=2            85.0%  [82.7%, 87.3%]  ['85.0%', '85.3%', '84.7%']
S1 Consensus        73.2%  [70.3%, 76.1%]  ['74.7%', '70.7%', '74.3%']
Opus Standard       82.4%  [79.9%, 84.9%]  ['81.7%', '83.0%', '82.7%']

McNemar exact test (pooled 3 seeds, N=900 pairs)
  DASE W=2 vs Opus Standard
    p=0.1149 ns  |  DASE W=2 wins=109, Opus Standard wins=86  (of 900 pairs)
  DASE W=2 vs S1 Consensus
    p=0.0000 ***  |  DASE W=2 wins=122, S1 Consensus wins=16  (of 900 pairs)
  S1 Consensus vs Opus Standard
    p=0.0000 ***  |  S1 Consensus wins=87, Opus Standard wins=170  (of 900 pairs)

Per-problem agreement (averaged across seeds)
  Both correct:       226 / 300
  DASE only correct:   31 / 300
  Opus only correct:   28 / 300
  Both wrong:          15 / 300

DASE commit reason breakdown (pooled across 3 seeds)
  left_wall                  465  (51.7%)
  right_wall                 435  (48.3%)

  

In [0]:
"""
DASE-Spatial (W=2) vs Claude Opus 4.6 Standard — AIME 2010-2026 (N=300)
7-panel visual dashboard · 3 seeds each · bootstrapped 95% CI · McNemar exact

Usage:
    python aime300_visual.py

Expected files in the same directory:
    DASE_Bellman_R300AIME_ai_2_L15_alpha0_2.csv
    DASE_Bellman_R300AIME_ai2_2_L15_alpha0_2.csv
    DASE_Bellman_R300AIME_ai3_2_L15_alpha0_2.csv
    DASE_Bellman_H300AIME_ai_2_L15_alpha0_2.jsonl   (optional — not used in plot)
    DASE_Bellman_H300AIME_ai2_2_L15_alpha0_2.jsonl  (optional)
    DASE_Bellman_H300AIME_ai3_2_L15_alpha0_2.jsonl  (optional)
    Opus_AIME300_1.csv
    Opus_AIME300_2.csv
    Opus_AIME300_3.csv

Output:
    AIME300_visual.png
"""

import csv
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as ticker
from scipy.stats import binomtest

# ── File paths ────────────────────────────────────────────────────────────────

DASE_CSV = [
    "DASE_Bellman_R300AIME_ai_2_L15_alpha0_2.csv",
    "DASE_Bellman_R300AIME_ai2_2_L15_alpha0_2.csv",
    "DASE_Bellman_R300AIME_ai3_2_L15_alpha0_2.csv",
]
OPUS_CSV = [
    "Opus_AIME300_1.csv",
    "Opus_AIME300_2.csv",
    "Opus_AIME300_3.csv",
]

# ── Palette ───────────────────────────────────────────────────────────────────
C_BG    = '#FFFFFF'
C_PANEL = '#F4F7FA'
C_GRID  = '#D1D9E0'
C_TEXT  = '#121212'
C_DIM   = '#57606A'

C_DASE  = '#2ECC8E'
C_OPUS  = '#5B9BD5'
C_S1    = '#4A90E2'
C_GOLD  = '#D4A017'
C_RIGHT = '#2ECC8E'
C_LEFT  = '#E05C5C'

# ── Helpers ───────────────────────────────────────────────────────────────────

def read_csv(path):
    with open(path, newline='', encoding='utf-8') as f:
        return list(csv.DictReader(f))

def to_acc(val):
    return 1 if str(val).strip() in ('1', '1.0', 'True', 'true') else 0

def bootstrap_ci(seeds, n_boot=50_000):
    """Resample problems within each seed, average across seeds → 95% CI."""
    boot = [
        np.mean([np.mean(np.random.choice(s, len(s), replace=True)) for s in seeds])
        for _ in range(n_boot)
    ]
    m = np.mean([np.mean(s) for s in seeds])
    return m, np.percentile(boot, 2.5), np.percentile(boot, 97.5)

def mcnemar_pooled(seeds_a, seeds_b):
    """McNemar exact test pooled across seeds (N = n_seeds × n_problems pairs)."""
    n10 = n01 = 0
    for sa, sb in zip(seeds_a, seeds_b):
        for a, b in zip(sa, sb):
            if a == 1 and b == 0: n10 += 1
            elif a == 0 and b == 1: n01 += 1
    disc = n10 + n01
    if disc == 0:
        return 1.0, n10, n01
    p = binomtest(n10, disc, 0.5, alternative='two-sided').pvalue
    return p, n10, n01

def sig_label(p):
    if   p < 0.001: return '*** p<0.001'
    elif p < 0.01:  return f'** p={p:.3f}'
    elif p < 0.05:  return f'* p={p:.3f}'
    else:           return f'ns p={p:.3f}'

def styled_ax(ax):
    ax.set_facecolor(C_PANEL)
    ax.tick_params(colors=C_DIM, labelsize=8.5)
    ax.spines[:].set_color(C_GRID)
    ax.spines[:].set_linewidth(0.8)
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)
    ax.grid(axis='y', color=C_GRID, linewidth=0.6, linestyle='--', alpha=0.7)
    ax.set_axisbelow(True)

def bracket(ax, x1, x2, y, label, col=C_DIM):
    h = 0.012
    ax.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.2, color=col)
    ax.text((x1+x2)/2, y+h+0.004, label,
            ha='center', va='bottom', fontsize=8, color=col)

# ── Load data ─────────────────────────────────────────────────────────────────

np.random.seed(42)

dase_seeds = [[to_acc(r['DASE_Acc']) for r in read_csv(p)] for p in DASE_CSV]
s1_seeds   = [[to_acc(r['S1_Acc'])   for r in read_csv(p)] for p in DASE_CSV]
opus_seeds = [[to_acc(r['FM_Acc'])   for r in read_csv(p)] for p in OPUS_CSV]

N = len(dase_seeds[0])
assert all(len(s) == N for s in dase_seeds + s1_seeds + opus_seeds), \
    "All seeds must have the same number of problems"
print(f"Loaded {N} problems × {len(dase_seeds)} seeds per method")

# ── Compute statistics ────────────────────────────────────────────────────────

# Bootstrapped means + CIs
dase_m, dase_lo, dase_hi = bootstrap_ci(dase_seeds)
opus_m, opus_lo, opus_hi = bootstrap_ci(opus_seeds)
s1_m,   s1_lo,   s1_hi   = bootstrap_ci(s1_seeds)

# McNemar tests
p_dv_o, n10_dvo, n01_dvo = mcnemar_pooled(dase_seeds, opus_seeds)
p_dv_s, _,       _        = mcnemar_pooled(dase_seeds, s1_seeds)
p_sv_o, _,       _        = mcnemar_pooled(s1_seeds,   opus_seeds)

# Per-seed accuracies (for jitter dots)
dase_sa = [np.mean(s) for s in dase_seeds]
opus_sa = [np.mean(s) for s in opus_seeds]
s1_sa   = [np.mean(s) for s in s1_seeds]

# Running accuracy (mean across seeds, cumulative)
dase_run = np.cumsum(np.mean(dase_seeds, axis=0)) / (np.arange(N) + 1)
opus_run = np.cumsum(np.mean(opus_seeds, axis=0)) / (np.arange(N) + 1)
s1_run   = np.cumsum(np.mean(s1_seeds,   axis=0)) / (np.arange(N) + 1)

# Per-problem means (majority-vote outcome across seeds)
dase_prob = np.mean(dase_seeds, axis=0)
opus_prob = np.mean(opus_seeds, axis=0)
dase_cor  = dase_prob >= 0.5
opus_cor  = opus_prob >= 0.5
both_cor   = int(np.sum( dase_cor &  opus_cor))
dase_only  = int(np.sum( dase_cor & ~opus_cor))
opus_only  = int(np.sum(~dase_cor &  opus_cor))
both_wrong = int(np.sum(~dase_cor & ~opus_cor))

# AIME 2026 subset (first 30 problems)
dase_26 = [s[:30] for s in dase_seeds]
opus_26 = [s[:30] for s in opus_seeds]
s1_26   = [s[:30] for s in s1_seeds]
dase_26_m = np.mean([np.mean(s) for s in dase_26])
opus_26_m = np.mean([np.mean(s) for s in opus_26])
s1_26_m   = np.mean([np.mean(s) for s in s1_26])
p_2026, n10_26, _ = mcnemar_pooled(dase_26, opus_26)

# Commit reason accuracy + steps (from CSV)
commit_acc   = {'right_wall': [], 'left_wall': []}
commit_steps = {'right_wall': [], 'left_wall': []}
all_steps    = []
for path in DASE_CSV:
    for row in read_csv(path):
        cr = row.get('Commit_Reason', '')
        if cr in commit_acc:
            commit_acc[cr].append(to_acc(row['DASE_Acc']))
            commit_steps[cr].append(int(row.get('Steps', 0)))
        all_steps.append(int(row.get('Steps', 0)))

commit_acc_right = float(np.mean(commit_acc['right_wall']))
commit_acc_left  = float(np.mean(commit_acc['left_wall']))
n_right = len(commit_acc['right_wall'])
n_left  = len(commit_acc['left_wall'])

# Print summary
print(f"\nDASE W=2:      {dase_m:.1%} [{dase_lo:.1%}, {dase_hi:.1%}]  seeds={[f'{x:.1%}' for x in dase_sa]}")
print(f"Opus Standard: {opus_m:.1%} [{opus_lo:.1%}, {opus_hi:.1%}]  seeds={[f'{x:.1%}' for x in opus_sa]}")
print(f"S1 Consensus:  {s1_m:.1%}  [{s1_lo:.1%}, {s1_hi:.1%}]  seeds={[f'{x:.1%}' for x in s1_sa]}")
print(f"\nMcNemar DASE vs Opus: p={p_dv_o:.4f} {sig_label(p_dv_o)}  (DASE wins={n10_dvo}, Opus wins={n01_dvo})")
print(f"McNemar DASE vs S1:   p={p_dv_s:.4f} {sig_label(p_dv_s)}")
print(f"\nAIME 2026: DASE={dase_26_m:.1%}  Opus={opus_26_m:.1%}  p={p_2026:.4f} {sig_label(p_2026)}")
print(f"Commit right_wall: {commit_acc_right:.1%} (n={n_right//3})  left_wall: {commit_acc_left:.1%} (n={n_left//3})")

# ── Figure ────────────────────────────────────────────────────────────────────

fig = plt.figure(figsize=(16, 12), facecolor=C_BG)
gs  = gridspec.GridSpec(
    3, 4, figure=fig,
    hspace=0.52, wspace=0.38,
    left=0.06, right=0.97,
    top=0.91,  bottom=0.07,
)

# ── Panel 1 — Main bar chart ──────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, :2])
styled_ax(ax1)

p1_labels = ['S1 Consensus\n(DASE Rd 1)', 'Opus 4.6\nStandard', 'DASE-Spatial\n(W=2)']
p1_means  = [s1_m,   opus_m,   dase_m]
p1_cis    = [(s1_lo, s1_hi), (opus_lo, opus_hi), (dase_lo, dase_hi)]
p1_colors = [C_S1,   C_OPUS,   C_DASE]
p1_seeds  = [s1_sa,  opus_sa,  dase_sa]

x1 = np.arange(3)
bars1 = ax1.bar(x1, p1_means, width=0.52, color=p1_colors, zorder=3, edgecolor='none')
ax1.errorbar(x1, p1_means,
             yerr=[[m - lo for m, (lo, hi) in zip(p1_means, p1_cis)],
                   [hi - m for m, (lo, hi) in zip(p1_means, p1_cis)]],
             fmt='none', color=C_TEXT,
             capsize=5, capthick=1.3, elinewidth=1.3, zorder=4)

rng = np.random.default_rng(7)
for i, sa in enumerate(p1_seeds):
    jitter = rng.uniform(-0.12, 0.12, len(sa))
    ax1.scatter(x1[i] + jitter, sa, color='white', s=36,
                edgecolors=p1_colors[i], linewidths=1.5, zorder=5)

for bar, m in zip(bars1, p1_means):
    ax1.text(bar.get_x() + bar.get_width()/2, m - 0.028,
             f'{m:.1%}', ha='center', va='top',
             fontsize=12, fontweight='bold', color='white', zorder=6)

top1 = max(hi for _, hi in p1_cis) + 0.03
bracket(ax1, 0, 2, top1 + 0.00,  sig_label(p_dv_s), C_DASE)
bracket(ax1, 1, 2, top1 + 0.055, sig_label(p_dv_o), C_DIM)

ax1.set_xticks(x1)
ax1.set_xticklabels(p1_labels, fontsize=9, color=C_TEXT)
ax1.yaxis.set_major_formatter(ticker.PercentFormatter(xmax=1, decimals=0))
ax1.set_ylim(0.5, top1 + 0.11)
ax1.set_yticks(np.arange(0.5, 1.01, 0.1))
ax1.axhline(dase_m, color=C_DASE, lw=0.7, ls='--', alpha=0.35, zorder=2)
ax1.tick_params(axis='x', length=0)
ax1.set_ylabel('Mean Accuracy', color=C_DIM, fontsize=8.5)
ax1.set_title('Accuracy · N=300 · 3 seeds each', color=C_TEXT, fontsize=10, pad=8, loc='left')

# ── Panel 2 — Running accuracy ────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 2:])
styled_ax(ax2)
ax2.grid(axis='both', color=C_GRID, linewidth=0.6, linestyle='--', alpha=0.7)

probs = np.arange(1, N + 1)
ax2.plot(probs, s1_run,   color=C_S1,  lw=1.4, label='S1 Consensus',  alpha=0.8)
ax2.plot(probs, opus_run, color=C_OPUS, lw=1.8, label='Opus Standard')
ax2.plot(probs, dase_run, color=C_DASE, lw=2.2, label='DASE W=2', zorder=4)
ax2.fill_between(probs, opus_run, dase_run,
                 where=dase_run > opus_run,
                 color=C_DASE, alpha=0.10, zorder=2)
ax2.axvline(30, color=C_GOLD, lw=1.0, ls=':', alpha=0.8)
ax2.text(31, 0.545, '2026→', color=C_GOLD, fontsize=7.5, va='bottom')

ax2.yaxis.set_major_formatter(ticker.PercentFormatter(xmax=1, decimals=0))
ax2.set_ylim(0.52, 0.95)
ax2.set_xlim(1, N)
ax2.set_xlabel('Problem index (cumulative)', color=C_DIM, fontsize=8.5)
ax2.set_title('Running accuracy across 300 problems', color=C_TEXT, fontsize=10, pad=8, loc='left')
ax2.legend(fontsize=8, framealpha=0, labelcolor=C_TEXT, loc='lower right')

# ── Panel 3 — Outcome breakdown ───────────────────────────────────────────────
ax3 = fig.add_subplot(gs[1, :2])
styled_ax(ax3)

p3_cats   = ['Both\nCorrect', 'DASE\nOnly', 'Opus\nOnly', 'Both\nWrong']
p3_counts = [both_cor, dase_only, opus_only, both_wrong]
p3_colors = [C_DASE,  C_DASE,    C_OPUS,    C_LEFT]
p3_alphas = [0.9,     1.0,       1.0,       0.55]

x3 = np.arange(4)
for i, (c, col, alp) in enumerate(zip(p3_counts, p3_colors, p3_alphas)):
    ax3.bar(x3[i], c, width=0.55, color=col, alpha=alp, zorder=3)
    ax3.text(x3[i], c + 2, f'{c}\n({c/N:.0%})',
             ha='center', va='bottom', fontsize=9,
             color=C_TEXT, fontweight='bold')

ax3.set_xticks(x3)
ax3.set_xticklabels(p3_cats, color=C_TEXT, fontsize=9)
ax3.set_ylim(0, 260)
ax3.set_ylabel('Problems', color=C_DIM, fontsize=8.5)
ax3.tick_params(axis='x', length=0)
ax3.grid(axis='x', visible=False)
ax3.set_title('Per-problem outcome (majority across seeds)', color=C_TEXT, fontsize=10, pad=8, loc='left')
ax3.annotate(f'Net DASE gain: +{dase_only - opus_only} problems',
             xy=(0.98, 0.93), xycoords='axes fraction',
             ha='right', color=C_DASE, fontsize=9, fontweight='bold')

# ── Panel 4 — Commit reason accuracy ─────────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 2])
styled_ax(ax4)

p4_accs   = [commit_acc_right, commit_acc_left]
p4_ns     = [n_right, n_left]
p4_labels = ['Right Wall\n(Consensus)', 'Left Wall\n(Fragmented)']
p4_colors = [C_RIGHT, C_LEFT]

bars4 = ax4.bar([0, 1], p4_accs, width=0.52, color=p4_colors, zorder=3)
for i, (bar, a, n) in enumerate(zip(bars4, p4_accs, p4_ns)):
    ax4.text(bar.get_x() + bar.get_width()/2, a - 0.028,
             f'{a:.1%}', ha='center', va='top',
             fontsize=11, fontweight='bold', color='white')
    ax4.text(bar.get_x() + bar.get_width()/2, 0.04,
             f'n={n//3}', ha='center', va='bottom',
             fontsize=7.5, color=C_DIM)

ax4.set_xticks([0, 1])
ax4.set_xticklabels(p4_labels, color=C_TEXT, fontsize=8.5)
ax4.yaxis.set_major_formatter(ticker.PercentFormatter(xmax=1, decimals=0))
ax4.set_ylim(0, 1.08)
ax4.tick_params(axis='x', length=0)
ax4.grid(axis='x', visible=False)
ax4.set_title('Accuracy by commit type', color=C_TEXT, fontsize=10, pad=8, loc='left')

# ── Panel 5 — Steps histogram ─────────────────────────────────────────────────
ax5 = fig.add_subplot(gs[1, 3])
styled_ax(ax5)

step_arr = np.array(all_steps)
bins5    = np.arange(0.5, step_arr.max() + 1.5)
n5, _, _ = ax5.hist(step_arr, bins=bins5, color=C_DASE, alpha=0.85,
                     edgecolor='none', zorder=3)
ax5.axvline(np.mean(step_arr), color=C_GOLD, lw=1.5, ls='--')
ax5.text(np.mean(step_arr) + 0.15, n5.max() * 0.92,
         f'μ={np.mean(step_arr):.1f}', color=C_GOLD, fontsize=8)

ax5.set_xlabel('Steps to commit', color=C_DIM, fontsize=8.5)
ax5.set_ylabel('Count', color=C_DIM, fontsize=8.5)
ax5.set_title('DASE stopping distribution', color=C_TEXT, fontsize=10, pad=8, loc='left')

# ── Panel 6 — AIME 2026 spotlight ────────────────────────────────────────────
ax6 = fig.add_subplot(gs[2, :2])
styled_ax(ax6)

p6_labels = ['S1 Consensus\n(DASE Rd 1)', 'Opus 4.6\nStandard', 'DASE-Spatial\n(W=2)']
p6_means  = [s1_26_m, opus_26_m, dase_26_m]
p6_colors = [C_S1, C_OPUS, C_DASE]

bars6 = ax6.bar(np.arange(3), p6_means, width=0.52, color=p6_colors, zorder=3, edgecolor='none')
for bar, m in zip(bars6, p6_means):
    ax6.text(bar.get_x() + bar.get_width()/2, m - 0.030,
             f'{m:.1%}', ha='center', va='top',
             fontsize=12, fontweight='bold', color='white', zorder=6)

bracket(ax6, 1, 2, max(p6_means) + 0.04, sig_label(p_2026), C_DASE)

ax6.set_xticks([0, 1, 2])
ax6.set_xticklabels(p6_labels, fontsize=9, color=C_TEXT)
ax6.yaxis.set_major_formatter(ticker.PercentFormatter(xmax=1, decimals=0))
ax6.set_ylim(0.4, max(p6_means) + 0.13)
ax6.tick_params(axis='x', length=0)
ax6.set_ylabel('Mean Accuracy', color=C_DIM, fontsize=8.5)
ax6.set_title('AIME 2026 spotlight (N=30, first 30 problems)', color=C_TEXT, fontsize=10, pad=8, loc='left')

# ── Panel 7 — Per-problem scatter ────────────────────────────────────────────
ax7 = fig.add_subplot(gs[2, 2:])
styled_ax(ax7)
ax7.grid(axis='both', color=C_GRID, linewidth=0.6, linestyle='--', alpha=0.7)

rng2 = np.random.default_rng(42)
jit  = rng2.uniform(-0.02, 0.02, N)
col_scatter = np.where(dase_prob > opus_prob, C_DASE,
              np.where(opus_prob > dase_prob, C_OPUS, C_DIM))
ax7.scatter(opus_prob + jit, dase_prob + jit,
            c=col_scatter, s=14, alpha=0.7, zorder=3)
ax7.plot([0, 1], [0, 1], color=C_DIM, lw=1.0, ls='--', alpha=0.6)

above = int(np.sum(dase_prob > opus_prob))
below = int(np.sum(dase_prob < opus_prob))
ax7.text(0.04, 0.93, f'DASE wins: {above}', transform=ax7.transAxes,
         color=C_DASE, fontsize=8.5, fontweight='bold')
ax7.text(0.04, 0.86, f'Opus wins: {below}', transform=ax7.transAxes,
         color=C_OPUS, fontsize=8.5, fontweight='bold')
ax7.text(0.04, 0.79, f'Equal: {N - above - below}', transform=ax7.transAxes,
         color=C_DIM, fontsize=8.5)

ax7.xaxis.set_major_formatter(ticker.PercentFormatter(xmax=1, decimals=0))
ax7.yaxis.set_major_formatter(ticker.PercentFormatter(xmax=1, decimals=0))
ax7.set_xlim(-0.05, 1.05)
ax7.set_ylim(-0.05, 1.05)
ax7.set_xlabel('Opus Standard accuracy (per problem)', color=C_DIM, fontsize=8.5)
ax7.set_ylabel('DASE W=2 accuracy (per problem)', color=C_DIM, fontsize=8.5)
ax7.set_title('Per-problem: DASE vs Opus (averaged across 3 seeds)', color=C_TEXT, fontsize=10, pad=8, loc='left')

# ── Master title & footer ─────────────────────────────────────────────────────
fig.text(0.5, 0.965,
         'DASE-Spatial (W=2)  vs  Claude Opus 4.6 Standard',
         ha='center', va='top', color=C_TEXT,
         fontsize=17, fontweight='bold')
fig.text(0.5, 0.945,
         'AIME 2010–2026  ·  N=300  ·  3 seeds per method  ·  '
         '95% bootstrap CI  ·  McNemar exact test (N=900 pairs)',
         ha='center', va='top', color=C_DIM, fontsize=9)

# ── Save ──────────────────────────────────────────────────────────────────────
out = 'AIME300_visual.png'
plt.savefig(out, dpi=170, bbox_inches='tight', facecolor=C_BG)
print(f"\nSaved → {out}")

Loaded 300 problems × 3 seeds per method

DASE W=2:      85.0% [82.7%, 87.3%]  seeds=['85.0%', '85.3%', '84.7%']
Opus Standard: 82.4% [79.9%, 84.9%]  seeds=['81.7%', '83.0%', '82.7%']
S1 Consensus:  73.2%  [70.3%, 76.1%]  seeds=['74.7%', '70.7%', '74.3%']

McNemar DASE vs Opus: p=0.1149 ns p=0.115  (DASE wins=109, Opus wins=86)
McNemar DASE vs S1:   p=0.0000 *** p<0.001

AIME 2026: DASE=87.8%  Opus=62.2%  p=0.0000 *** p<0.001
Commit right_wall: 95.2% (n=145)  left_wall: 75.5% (n=155)

Saved → AIME300_visual.png


In [0]:
"""
AIME 2010-2026 (N=300) — Paper Figure 2
Three panels:
  (a) Running accuracy across 300 problems
  (b) Accuracy by commit type (right wall vs left wall)
  (c) Per-problem scatter: DASE vs Opus

Academic style — white background, serif font — matches aime2026_paper_figure.py

Usage (run from directory containing all CSV files):
    python aime300_paper_figure.py

Output: AIME300_paper_figure.png
"""

import csv
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from scipy.stats import binomtest

# ── File paths ────────────────────────────────────────────────────────────────

DASE_CSV = [
    "DASE_Bellman_R300AIME_ai_2_L15_alpha0_2.csv",
    "DASE_Bellman_R300AIME_ai2_2_L15_alpha0_2.csv",
    "DASE_Bellman_R300AIME_ai3_2_L15_alpha0_2.csv",
]
OPUS_CSV = [
    "Opus_AIME300_1.csv",
    "Opus_AIME300_2.csv",
    "Opus_AIME300_3.csv",
]

# ── Colors — consistent with aime2026_paper_figure.py ────────────────────────

C_OPUS_STD = '#E8A09A'   # light salmon
C_S1       = '#AAAAAA'   # neutral gray
C_DASE2    = '#6DB56D'   # medium green
C_RIGHT    = '#2A8A2A'   # dark green  (right wall = consensus)
C_LEFT     = '#C44E52'   # deep red    (left wall  = fragmented)
C_GOLD     = '#D4A017'   # AIME 2026 divider

# ── Matplotlib style ──────────────────────────────────────────────────────────

plt.rcParams.update({
    'font.family':       'serif',
    'font.serif':        ['DejaVu Serif', 'Times New Roman', 'Georgia'],
    'font.size':         10,
    'axes.titlesize':    10,
    'axes.labelsize':    9.5,
    'xtick.labelsize':   9,
    'ytick.labelsize':   9,
    'axes.linewidth':    0.8,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'grid.color':        '#E0E0E0',
    'grid.linewidth':    0.6,
    'grid.linestyle':    '--',
    'figure.facecolor':  'white',
    'axes.facecolor':    '#FAFAFA',
    'savefig.facecolor': 'white',
    'legend.framealpha': 0.9,
    'legend.edgecolor':  '#CCCCCC',
    'legend.fontsize':   8.5,
})

# ── Helpers ───────────────────────────────────────────────────────────────────

def read_csv(path):
    with open(path, newline='', encoding='utf-8') as f:
        return list(csv.DictReader(f))

def to_acc(val):
    return 1 if str(val).strip() in ('1', '1.0', 'True', 'true') else 0

def bootstrap_ci(seeds, n_boot=50_000):
    """95% CI by resampling problems within each seed, averaging across seeds."""
    boot = [
        np.mean([np.mean(np.random.choice(s, len(s), replace=True)) for s in seeds])
        for _ in range(n_boot)
    ]
    m = np.mean([np.mean(s) for s in seeds])
    return m, np.percentile(boot, 2.5), np.percentile(boot, 97.5)

def bootstrap_ci_flat(vals, n_boot=50_000):
    """95% CI for a flat list of binary outcomes."""
    boot = [np.mean(np.random.choice(vals, len(vals), replace=True))
            for _ in range(n_boot)]
    return np.mean(vals), np.percentile(boot, 2.5), np.percentile(boot, 97.5)

def mcnemar_pooled(seeds_a, seeds_b):
    """McNemar exact test pooled across seeds."""
    n10 = n01 = 0
    for sa, sb in zip(seeds_a, seeds_b):
        for a, b in zip(sa, sb):
            if a == 1 and b == 0: n10 += 1
            elif a == 0 and b == 1: n01 += 1
    disc = n10 + n01
    if disc == 0:
        return 1.0, n10, n01
    p = binomtest(n10, disc, 0.5, alternative='two-sided').pvalue
    return p, n10, n01

def sig_label(p):
    if   p < 0.001: return '*** p<0.001'
    elif p < 0.01:  return f'**  p={p:.3f}'
    elif p < 0.05:  return f'*   p={p:.3f}'
    else:           return f'ns  p={p:.3f}'

def style_ax(ax, both_axes=False):
    ax.set_facecolor('#FAFAFA')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.8)
    ax.spines['bottom'].set_linewidth(0.8)
    ax.grid(color='#E0E0E0', linewidth=0.6, linestyle='--',
            axis='both' if both_axes else 'y')
    ax.set_axisbelow(True)

# ── Load data ─────────────────────────────────────────────────────────────────

np.random.seed(42)

dase_seeds = [[to_acc(r['DASE_Acc']) for r in read_csv(p)] for p in DASE_CSV]
s1_seeds   = [[to_acc(r['S1_Acc'])   for r in read_csv(p)] for p in DASE_CSV]
opus_seeds = [[to_acc(r['FM_Acc'])   for r in read_csv(p)] for p in OPUS_CSV]

N = len(dase_seeds[0])
assert all(len(s) == N for s in dase_seeds + s1_seeds + opus_seeds)
print(f"Loaded {N} problems × {len(dase_seeds)} seeds per method")

# ── Compute statistics ────────────────────────────────────────────────────────

dase_m, dase_lo, dase_hi = bootstrap_ci(dase_seeds)
opus_m, opus_lo, opus_hi = bootstrap_ci(opus_seeds)
s1_m,   s1_lo,   s1_hi   = bootstrap_ci(s1_seeds)

p_dv_o, n10_dvo, n01_dvo = mcnemar_pooled(dase_seeds, opus_seeds)
p_dv_s, _,       _        = mcnemar_pooled(dase_seeds, s1_seeds)

dase_run = np.cumsum(np.mean(dase_seeds, axis=0)) / (np.arange(N) + 1)
opus_run = np.cumsum(np.mean(opus_seeds, axis=0)) / (np.arange(N) + 1)
s1_run   = np.cumsum(np.mean(s1_seeds,   axis=0)) / (np.arange(N) + 1)

dase_prob = np.mean(dase_seeds, axis=0)
opus_prob = np.mean(opus_seeds, axis=0)

commit_acc = {'right_wall': [], 'left_wall': []}
for path in DASE_CSV:
    for row in read_csv(path):
        cr = row.get('Commit_Reason', '')
        if cr in commit_acc:
            commit_acc[cr].append(to_acc(row['DASE_Acc']))

commit_n_right = len(commit_acc['right_wall']) // 3
commit_n_left  = len(commit_acc['left_wall'])  // 3
right_m, right_lo, right_hi = bootstrap_ci_flat(commit_acc['right_wall'])
left_m,  left_lo,  left_hi  = bootstrap_ci_flat(commit_acc['left_wall'])

print(f"\nDASE W=2:      {dase_m:.1%} [{dase_lo:.1%}, {dase_hi:.1%}]")
print(f"Opus Standard: {opus_m:.1%} [{opus_lo:.1%}, {opus_hi:.1%}]")
print(f"McNemar DASE vs Opus: {sig_label(p_dv_o)} "
      f"(DASE>Opus: {n10_dvo}, Opus>DASE: {n01_dvo})")
print(f"Right wall: {right_m:.1%} [{right_lo:.1%}, {right_hi:.1%}]  n={commit_n_right}")
print(f"Left  wall: {left_m:.1%}  [{left_lo:.1%}, {left_hi:.1%}]   n={commit_n_left}")

# ── Figure layout ─────────────────────────────────────────────────────────────
# fig.add_axes() with explicit coordinates prevents matplotlib from
# allowing rotated y-labels to bleed between panels.
#
# Panel positions (left, bottom, width, height) in figure-fraction units:
#   ax_run:     0.07, 0.16, 0.44, 0.67   — wide running-accuracy panel
#   ax_commit:  0.57, 0.16, 0.12, 0.67   — narrow commit-reason panel
#   ax_scatter: 0.77, 0.16, 0.21, 0.67   — scatter panel
#
# Gap (a)→(b): 0.06   Gap (b)→(c): 0.08  — generous to prevent label bleed

fig = plt.figure(figsize=(13.5, 4.8), facecolor='white')

fig.text(0.5, 0.975,
         'DASE-Spatial (W=2)  vs.  Claude Opus 4.6 Standard'
         '  —  AIME 2010–2026  (N=300)',
         ha='center', va='top', fontsize=11.5,
         fontweight='bold', color='#111111')

ax_run     = fig.add_axes([0.07, 0.16, 0.44, 0.67])
ax_commit  = fig.add_axes([0.57, 0.16, 0.12, 0.67])
ax_scatter = fig.add_axes([0.77, 0.16, 0.21, 0.67])

# ═══════════════════════════════════════════════════════════════════
# Panel (a) — Running accuracy
# ═══════════════════════════════════════════════════════════════════
style_ax(ax_run, both_axes=True)

probs = np.arange(1, N + 1)
ax_run.plot(probs, s1_run,   color=C_S1,      lw=1.4,
            label='S1 Consensus (DASE Rd. 1)', alpha=0.85)
ax_run.plot(probs, opus_run, color=C_OPUS_STD, lw=1.8,
            label='Opus 4.6 Standard')
ax_run.plot(probs, dase_run, color=C_DASE2,    lw=2.2,
            label='DASE-Spatial (W=2)', zorder=4)
ax_run.fill_between(probs, opus_run, dase_run,
                    where=dase_run > opus_run,
                    color=C_DASE2, alpha=0.10, zorder=2)

ax_run.axvline(30, color=C_GOLD, lw=1.0, ls=':', alpha=0.9)
ax_run.text(32, 0.545, 'AIME 2026 →', color=C_GOLD, fontsize=7.5, va='bottom')

# Final value labels placed INSIDE axes (x=284) — no clip_on=False needed
for val, col, dy in [(dase_run[-1], C_DASE2,    0.010),
                      (opus_run[-1], C_OPUS_STD, -0.020),
                      (s1_run[-1],   C_S1,        0.010)]:
    ax_run.text(284, val + dy, f'{val:.1%}',
                fontsize=8, color=col, fontweight='bold',
                va='center', ha='right')

ax_run.yaxis.set_major_formatter(ticker.PercentFormatter(xmax=1, decimals=0))
ax_run.set_ylim(0.52, 0.96)
ax_run.set_xlim(1, 300)
ax_run.set_xlabel('Problem index (cumulative)', fontsize=9.5)
ax_run.set_ylabel('Running mean accuracy', fontsize=9.5)
ax_run.set_title('(a)  Running accuracy  ·  N=300  ·  3 seeds per method',
                 fontsize=9.5, pad=6, loc='left')
ax_run.legend(loc='lower right', fontsize=8, framealpha=0.9)
ax_run.text(0.98, 0.09,
            f'DASE vs. Opus: {sig_label(p_dv_o)}\n'
            f'DASE vs. S1:   {sig_label(p_dv_s)}',
            transform=ax_run.transAxes, ha='right', va='bottom', fontsize=7.5,
            color='#444444',
            bbox=dict(fc='white', ec='#CCCCCC', pad=3, alpha=0.9))

# ═══════════════════════════════════════════════════════════════════
# Panel (b) — Accuracy by commit type
# ═══════════════════════════════════════════════════════════════════
style_ax(ax_commit)

accs    = [right_m, left_m]
lo_errs = [right_m - right_lo, left_m - left_lo]
hi_errs = [right_hi - right_m, left_hi - left_m]

bars = ax_commit.bar([0, 1], accs, width=0.52, color=[C_RIGHT, C_LEFT],
                     zorder=3, edgecolor='white', linewidth=0.5)
ax_commit.errorbar([0, 1], accs, yerr=[lo_errs, hi_errs],
                   fmt='none', color='#333333',
                   capsize=4, capthick=1.2, elinewidth=1.2, zorder=4)

for bar, a, n in zip(bars, accs, [commit_n_right, commit_n_left]):
    ax_commit.text(bar.get_x() + bar.get_width()/2, a - 0.030,
                   f'{a:.1%}', ha='center', va='top',
                   fontsize=10, fontweight='bold', color='white')
    ax_commit.text(bar.get_x() + bar.get_width()/2, 0.04,
                   f'n={n}', ha='center', va='bottom',
                   fontsize=8, color='#666666')

# Difference annotation — within bar width, no overflow
ax_commit.annotate('', xy=(0.75, left_m), xytext=(0.75, right_m),
                   arrowprops=dict(arrowstyle='<->', color='#555555', lw=1.0))
ax_commit.text(0.60, (right_m + left_m)/2,
               f'−{right_m - left_m:.1%}',
               ha='right', va='center', fontsize=8, color='#555555')

ax_commit.set_xticks([0, 1])
ax_commit.set_xticklabels(['Right Wall\n(Consensus)', 'Left Wall\n(Fragmented)'],
                           fontsize=8.5)
ax_commit.yaxis.set_major_formatter(ticker.PercentFormatter(xmax=1, decimals=0))
ax_commit.set_ylim(0, 1.12)
ax_commit.tick_params(axis='x', length=0)
ax_commit.set_ylabel('Accuracy', fontsize=9.5)
ax_commit.set_title('(b)  Accuracy by\ncommit type', fontsize=9.5, pad=6, loc='left')

# ═══════════════════════════════════════════════════════════════════
# Panel (c) — Per-problem scatter
# ═══════════════════════════════════════════════════════════════════
style_ax(ax_scatter, both_axes=True)

rng2 = np.random.default_rng(42)
jit  = rng2.uniform(-0.022, 0.022, N)

col_scatter = np.where(
    dase_prob > opus_prob, C_DASE2,
    np.where(opus_prob > dase_prob, C_OPUS_STD, '#AAAAAA')
)
ax_scatter.scatter(opus_prob + jit, dase_prob + jit,
                   c=col_scatter, s=13, alpha=0.75, zorder=3, linewidths=0)
ax_scatter.plot([0, 1], [0, 1], color='#AAAAAA', lw=1.0, ls='--', alpha=0.7)

above = int(np.sum(dase_prob > opus_prob))
below = int(np.sum(dase_prob < opus_prob))
equal = N - above - below

ax_scatter.text(0.05, 0.95, f'DASE wins: {above} ({above/N:.0%})',
                transform=ax_scatter.transAxes,
                color=C_DASE2, fontsize=8.5, fontweight='bold', va='top')
ax_scatter.text(0.05, 0.88, f'Opus wins: {below} ({below/N:.0%})',
                transform=ax_scatter.transAxes,
                color=C_OPUS_STD, fontsize=8.5, fontweight='bold', va='top')
ax_scatter.text(0.05, 0.81, f'Equal: {equal} ({equal/N:.0%})',
                transform=ax_scatter.transAxes, color='#888888', fontsize=8, va='top')

ax_scatter.xaxis.set_major_formatter(ticker.PercentFormatter(xmax=1, decimals=0))
ax_scatter.yaxis.set_major_formatter(ticker.PercentFormatter(xmax=1, decimals=0))
ax_scatter.set_xlim(-0.06, 1.06)
ax_scatter.set_ylim(-0.06, 1.06)
# Shorter labels — prevents y-label rotating into panel (b)
ax_scatter.set_xlabel('Opus 4.6 Standard accuracy\n(per problem)', fontsize=9)
ax_scatter.set_ylabel('DASE W=2 accuracy\n(per problem)', fontsize=9, labelpad=4)
ax_scatter.set_title('(c)  Per-problem outcomes\n(averaged across 3 seeds)',
                     fontsize=9.5, pad=6, loc='left')

# ── Footer ────────────────────────────────────────────────────────────────────

fig.text(0.5, 0.005,
         'DASE: 3 seeds · Opus: 3 seeds · 95% bootstrap CI · '
         'McNemar exact test pooled across seeds (N=900 pairs) · '
         'DASE ensemble: 3×GPT-OSS-120B + 2×Qwen3-80B-A3B · max_tokens=8,000',
         ha='center', fontsize=7.5, color='#777777', style='italic')

# ── Save ──────────────────────────────────────────────────────────────────────

out = 'AIME300_paper_figure.png'
plt.savefig(out, dpi=200, bbox_inches='tight', facecolor='white')
print(f"\nSaved → {out}")

Loaded 300 problems × 3 seeds per method

DASE W=2:      85.0% [82.7%, 87.3%]
Opus Standard: 82.4% [79.9%, 84.9%]
McNemar DASE vs Opus: ns  p=0.115 (DASE>Opus: 109, Opus>DASE: 86)
Right wall: 95.2% [93.1%, 97.0%]  n=145
Left  wall: 75.5%  [71.6%, 79.4%]   n=155

Saved → AIME300_paper_figure.png


In [0]:
"""
AIME 2026 paper figure — two panels:
  (a) accuracy bar chart with significance brackets
  (b) cost–accuracy frontier

Run from the directory containing all CSV files:
    python aime2026_paper_figure.py

Output: AIME2026_paper_figure.png
"""

import csv
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from scipy.stats import binomtest

# ── File paths ────────────────────────────────────────────────────────────────

DASE2_CSV = [
    "DASE_Bellman_R30AIME_ai_2_L15_alpha0_2.csv",     # seed 1
    "DASE_Bellman_R300AIME_ai_2_L15_alpha0_2.csv",    # seed 2 — first 30 rows used
    "DASE_Bellman_R300AIME_ai2_2_L15_alpha0_2.csv",   # seed 3 — first 30 rows used
]
DASE8_CSV = [
    "DASE_Bellman_R30AIME_ai_8_L15_alpha0_2.csv",
    "DASE_Bellman_R30AIME_ai2_8_L15_alpha0_2.csv",
    "DASE_Bellman_R30AIME_ai3_8_L15_alpha0_2.csv",
]
OPUS_STD_CSV = [
    "claude-opus-4-6_standard_AIME2026_1.csv",
    "claude-opus-4-6_standard_AIME2026_2.csv",
    "claude-opus-4-6_standard_AIME2026_3.csv",
]
OPUS_120K_CSV = [
    "Opus46_thinking_enabled_120k_AIME2026.csv",
    "Opus46_thinking_enabled_120k_AIME2026_2.csv",
    "Opus46_thinking_enabled_120k_AIME2026_3.csv",
]

# Cost per question (USD) — confirmed from billing data
COSTS = {
    "Opus Standard":  0.057,
    "S1 Consensus":   0.014,
    "DASE W=2":       0.041,
    "DASE W=8":       0.145,
    "Opus 120k High": 0.273,
}

# ── Colors ────────────────────────────────────────────────────────────────────

C_OPUS_STD  = '#E8A09A'   # light salmon — same family as Opus High
C_S1        = '#AAAAAA'   # neutral gray
C_DASE2     = '#6DB56D'   # medium green
C_DASE8     = '#2A8A2A'   # dark green
C_OPUS_HIGH = '#C44E52'   # deep red

# ── Helpers ───────────────────────────────────────────────────────────────────

def read_csv(path):
    with open(path, newline='', encoding='utf-8') as f:
        return list(csv.DictReader(f))

def to_acc(val):
    return 1 if str(val).strip() in ('1', '1.0', 'True', 'true') else 0

def bootstrap_ci(seeds, n_boot=50_000):
    """95% CI: resample problems within each seed, average across seeds."""
    boot = [
        np.mean([np.mean(np.random.choice(s, len(s), replace=True)) for s in seeds])
        for _ in range(n_boot)
    ]
    m = np.mean([np.mean(s) for s in seeds])
    return m, np.percentile(boot, 2.5), np.percentile(boot, 97.5)

def mcnemar_pooled(seeds_a, seeds_b):
    """McNemar exact test pooled across seeds."""
    n10 = n01 = 0
    for sa, sb in zip(seeds_a, seeds_b):
        for a, b in zip(sa, sb):
            if a == 1 and b == 0: n10 += 1
            elif a == 0 and b == 1: n01 += 1
    disc = n10 + n01
    if disc == 0:
        return 1.0, n10, n01
    p = binomtest(n10, disc, 0.5, alternative='two-sided').pvalue
    return p, n10, n01

def sig_label(p):
    if   p < 0.001: return '*** p < 0.001'
    elif p < 0.01:  return f'**  p = {p:.3f}'
    elif p < 0.05:  return f'*   p = {p:.3f}'
    else:           return f'ns  p = {p:.3f}'

def bracket(ax, x1, x2, y, label, col='#555555', lw=1.0):
    h = 0.011
    ax.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=lw, color=col, clip_on=False)
    ax.text((x1+x2)/2, y+h+0.003, label,
            ha='center', va='bottom', fontsize=7.5, color=col, clip_on=False)

# ── Load data ─────────────────────────────────────────────────────────────────

np.random.seed(42)

dase2_seeds = [
    [to_acc(r['DASE_Acc']) for r in read_csv(DASE2_CSV[0])],
    [to_acc(r['DASE_Acc']) for r in read_csv(DASE2_CSV[1])][:30],
    [to_acc(r['DASE_Acc']) for r in read_csv(DASE2_CSV[2])][:30],
]
s1_seeds = [
    [to_acc(r['S1_Acc']) for r in read_csv(DASE8_CSV[0])],
    [to_acc(r['S1_Acc']) for r in read_csv(DASE8_CSV[1])],
    [to_acc(r['S1_Acc']) for r in read_csv(DASE8_CSV[2])],
]
dase8_seeds = [
    [to_acc(r['DASE_Acc']) for r in read_csv(p)] for p in DASE8_CSV
]
opus_std_seeds = [
    [to_acc(r['FM_Acc']) for r in read_csv(p)] for p in OPUS_STD_CSV
]
opus_120k_seeds = [
    [to_acc(r['FM_Acc']) for r in read_csv(p)] for p in OPUS_120K_CSV
]

ALL_SEEDS = {
    "Opus Standard":  opus_std_seeds,
    "S1 Consensus":   s1_seeds,
    "DASE W=2":       dase2_seeds,
    "DASE W=8":       dase8_seeds,
    "Opus 120k High": opus_120k_seeds,
}

# ── Compute statistics ────────────────────────────────────────────────────────

results = {name: bootstrap_ci(seeds) for name, seeds in ALL_SEEDS.items()}
seed_accs = {name: [np.mean(s) for s in seeds] for name, seeds in ALL_SEEDS.items()}

# Print summary
print(f"{'Method':<18} {'Mean':>7}  {'95% CI':<22}  Seeds")
print("-" * 65)
for name, (m, lo, hi) in results.items():
    sa = [f"{x:.1%}" for x in seed_accs[name]]
    print(f"{name:<18} {m:.1%}   [{lo:.1%}, {hi:.1%}]   {sa}")

print("\nMcNemar exact (N=90 pairs):")
tests = [
    ("DASE W=2",       "Opus Standard"),
    ("DASE W=8",       "Opus Standard"),
    ("DASE W=8",       "DASE W=2"),
    ("DASE W=8",       "Opus 120k High"),
    ("Opus 120k High", "DASE W=2"),
]
p_vals = {}
for ka, kb in tests:
    p, n10, n01 = mcnemar_pooled(ALL_SEEDS[ka], ALL_SEEDS[kb])
    p_vals[f"{ka}_vs_{kb}"] = p
    stars = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
    print(f"  {ka} vs {kb}: p={p:.4f} {stars}  ({ka}>{kb}: {n10}, {kb}>{ka}: {n01})")

# ── Plot setup ────────────────────────────────────────────────────────────────

plt.rcParams.update({
    'font.family':      'serif',
    'font.serif':       ['DejaVu Serif', 'Times New Roman', 'Georgia'],
    'font.size':        10,
    'axes.titlesize':   10,
    'axes.labelsize':   9.5,
    'xtick.labelsize':  9,
    'ytick.labelsize':  9,
    'axes.linewidth':   0.8,
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'grid.color':       '#E0E0E0',
    'grid.linewidth':   0.6,
    'grid.linestyle':   '--',
    'figure.facecolor': 'white',
    'axes.facecolor':   '#FAFAFA',
    'savefig.facecolor':'white',
})

ORDER   = ['Opus Standard', 'S1 Consensus', 'DASE W=2', 'DASE W=8', 'Opus 120k High']
COLORS  = [C_OPUS_STD, C_S1, C_DASE2, C_DASE8, C_OPUS_HIGH]
XLABELS = ['Opus 4.6\nStandard', 'S1 Consensus\n(DASE Rd. 1)',
           'DASE\n(W=2)', 'DASE\n(W=8)', 'Opus 4.6\n120k Thinking']

means  = [results[k][0] for k in ORDER]
lo_err = [results[k][0] - results[k][1] for k in ORDER]
hi_err = [results[k][2] - results[k][0] for k in ORDER]
seeds  = [seed_accs[k] for k in ORDER]
costs  = [COSTS[k] for k in ORDER]

fig = plt.figure(figsize=(11, 5.0), facecolor='white')
ax1 = fig.add_axes([0.06, 0.17, 0.52, 0.66])
ax2 = fig.add_axes([0.64, 0.17, 0.33, 0.66])

for ax in (ax1, ax2):
    ax.set_facecolor('#FAFAFA')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.8)
    ax.spines['bottom'].set_linewidth(0.8)
    ax.grid(color='#E0E0E0', linewidth=0.6, linestyle='--')
    ax.set_axisbelow(True)

# ── Panel A — Bar chart ───────────────────────────────────────────────────────

x = np.arange(len(ORDER))
bars = ax1.bar(x, means, width=0.55, color=COLORS, zorder=3,
               edgecolor='white', linewidth=0.6)
ax1.errorbar(x, means, yerr=[lo_err, hi_err],
             fmt='none', color='#333333',
             capsize=4, capthick=1.2, elinewidth=1.2, zorder=4)

rng = np.random.default_rng(7)
for i, sa in enumerate(seeds):
    j = rng.uniform(-0.13, 0.13, len(sa))
    fc = '#EEEEEE' if COLORS[i] == C_S1 else 'white'
    ax1.scatter(x[i]+j, sa, color=fc, s=30,
                edgecolors=COLORS[i], linewidths=1.4, zorder=5)

txt_colors = ['#7A2020', '#444444', 'white', 'white', 'white']
for bar, m, tc in zip(bars, means, txt_colors):
    ax1.text(bar.get_x()+bar.get_width()/2, m - 0.025,
             f'{m:.1%}', ha='center', va='top',
             fontsize=9.5, fontweight='bold', color=tc, zorder=6)

top = max(m+e for m, e in zip(means, hi_err)) + 0.03
ymax = top + 0.165
bracket(ax1, 3, 4, top+0.000, sig_label(p_vals['DASE W=8_vs_Opus 120k High']), '#888888', 0.9)
bracket(ax1, 0, 3, top+0.058, sig_label(p_vals['DASE W=8_vs_Opus Standard']),  C_DASE8,  1.3)
bracket(ax1, 0, 2, top+0.110, sig_label(p_vals['DASE W=2_vs_Opus Standard']),  C_DASE2,  1.3)

ax1.axvline(1.5, color='#CCCCCC', lw=0.9, ls=':', zorder=2)
ax1.text(0.5,  ymax*0.999, 'Proprietary',
         ha='center', va='top', fontsize=7.5, color='#999999', style='italic')
ax1.text(3.0,  ymax*0.999, 'DASE (open-weight ensemble)',
         ha='center', va='top', fontsize=7.5, color='#2A8A2A', style='italic')

ax1.set_xticks(x)
ax1.set_xticklabels(XLABELS, fontsize=8.8)
ax1.yaxis.set_major_formatter(ticker.PercentFormatter(xmax=1, decimals=0))
ax1.set_ylim(0.38, ymax)
ax1.set_yticks(np.arange(0.4, 1.01, 0.1))
ax1.set_ylabel('Mean Accuracy  (N = 30)', fontsize=9.5)
ax1.set_xlabel('3 seeds · 95% bootstrap CI · McNemar exact test (N = 90 pairs)',
               fontsize=8, color='#666666', labelpad=6)
ax1.set_title('(a)  AIME 2026 accuracy by method', fontsize=9.5, pad=8, loc='left')

for i, (c, col) in enumerate(zip(costs, COLORS)):
    ax1.annotate(f'${c:.3f}/q',
                 xy=(i, 0.388), xycoords=('data', 'data'),
                 ha='center', va='top', fontsize=7.5, color='#777777')

# ── Panel B — Cost–accuracy frontier ─────────────────────────────────────────

markers = ['s', 'D', 'o', 'o', 's']
for name, col, mk in zip(ORDER, COLORS, markers):
    m, lo, hi = results[name]
    c = COSTS[name]
    ax2.errorbar(c, m, yerr=[[m-lo],[hi-m]],
                 fmt=mk, color=col, markersize=9,
                 capsize=4, capthick=1.2, elinewidth=1.2,
                 markeredgecolor='white', markeredgewidth=0.7,
                 zorder=4)

dc = [COSTS['DASE W=2'], COSTS['DASE W=8']]
dm = [results['DASE W=2'][0], results['DASE W=8'][0]]
ax2.plot(dc, dm, color=C_DASE8, lw=1.3, ls='--', alpha=0.5, zorder=3)

label_offsets = {
    'Opus Standard':  ( 0.006, -0.030),
    'S1 Consensus':   ( 0.006,  0.010),
    'DASE W=2':       (-0.003,  0.018),
    'DASE W=8':       ( 0.006,  0.008),
    'Opus 120k High': ( 0.006, -0.030),
}
short_names = {
    'Opus Standard':  'Opus Std.',
    'S1 Consensus':   'S1 Cons.',
    'DASE W=2':       'DASE (W=2)',
    'DASE W=8':       'DASE (W=8)',
    'Opus 120k High': 'Opus 120k',
}
for name, col in zip(ORDER, COLORS):
    m = results[name][0]
    c = COSTS[name]
    dx, dy = label_offsets[name]
    ax2.text(c+dx, m+dy, short_names[name],
             fontsize=8, color=col, fontweight='bold', va='center')

mid_c = (COSTS['DASE W=8'] + COSTS['Opus 120k High']) / 2
mid_m = (results['DASE W=8'][0] + results['Opus 120k High'][0]) / 2
ax2.annotate('Statistically\ntied (p = 0.250)',
             xy=(mid_c, mid_m), xytext=(0.10, 0.945),
             fontsize=7.5, color='#666666', ha='center',
             arrowprops=dict(arrowstyle='->', color='#999999', lw=0.9))

ax2.set_xlabel('Cost per question (USD)', fontsize=9.5)
ax2.set_ylabel('Mean Accuracy  (N = 30)', fontsize=9.5)
ax2.yaxis.set_major_formatter(ticker.PercentFormatter(xmax=1, decimals=0))
ax2.xaxis.set_major_formatter(ticker.FormatStrFormatter('$%.2f'))
ax2.set_ylim(0.50, 1.02)
ax2.set_xlim(-0.01, 0.32)
ax2.grid(axis='both')
ax2.set_title('(b)  Cost–accuracy frontier\nError bars: 95% bootstrap CI',
              fontsize=9.5, pad=6, loc='left')

# ── Title and footer ──────────────────────────────────────────────────────────

fig.text(0.5, 0.96,
         'DASE-Spatial vs. Claude Opus 4.6  —  AIME 2026  (N = 30, 3 seeds per method)',
         ha='center', va='top', fontsize=11.5,
         fontweight='bold', color='#111111')

fig.text(0.5, 0.005,
         'Dots: individual seed accuracies · '
         'Error bars: 95% bootstrap CI · '
         'McNemar exact test pooled across seeds (N = 90 pairs) · '
         'DASE ensemble: 3×GPT-OSS-120B + 2×Qwen3-80B-A3B',
         ha='center', fontsize=7.5, color='#777777', style='italic')

# ── Save ──────────────────────────────────────────────────────────────────────

out = 'AIME2026_paper_figure.png'
plt.savefig(out, dpi=200, bbox_inches='tight', facecolor='white')
print(f"\nSaved → {out}")

Method                Mean  95% CI                  Seeds
-----------------------------------------------------------------
Opus Standard      60.0%   [50.0%, 70.0%]   ['56.7%', '56.7%', '66.7%']
S1 Consensus       76.7%   [67.8%, 85.6%]   ['80.0%', '80.0%', '70.0%']
DASE W=2           85.6%   [77.8%, 92.2%]   ['80.0%', '86.7%', '90.0%']
DASE W=8           90.0%   [83.3%, 95.6%]   ['90.0%', '90.0%', '90.0%']
Opus 120k High     93.3%   [87.8%, 97.8%]   ['93.3%', '93.3%', '93.3%']

McNemar exact (N=90 pairs):
  DASE W=2 vs Opus Standard: p=0.0000 ***  (DASE W=2>Opus Standard: 24, Opus Standard>DASE W=2: 1)
  DASE W=8 vs Opus Standard: p=0.0000 ***  (DASE W=8>Opus Standard: 27, Opus Standard>DASE W=8: 0)
  DASE W=8 vs DASE W=2: p=0.1250 ns  (DASE W=8>DASE W=2: 4, DASE W=2>DASE W=8: 0)
  DASE W=8 vs Opus 120k High: p=0.2500 ns  (DASE W=8>Opus 120k High: 0, Opus 120k High>DASE W=8: 3)
  Opus 120k High vs DASE W=2: p=0.0156 *  (Opus 120k High>DASE W=2: 7, DASE W=2>Opus 120k High: 0)

Saved →